In [1]:
from datasets import load_dataset
from pathlib import Path
import sys
from torch.utils.data import DataLoader

In [2]:
dataset = load_dataset("ai-enthusiasm-community/KTVIC")

In [3]:
# structure of dataset 
dataset['train'][0]

{'image_uid': '000000000001',
 'caption_uid': ['000000000001_1',
  '000000000001_2',
  '000000000001_3',
  '000000000001_4',
  '000000000001_5'],
 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=695x456>,
 'caption_vi': ['ba chiếc thuyền đang di chuyển ở trên con sông',
  'có ba con thuyền đang di chuyển trên con sông',
  'trên dòng sông có ba con thuyền đang di chuyển',
  'ba con thuyền đang di chuyển bên một cánh đồng lúa',
  'ba chiếc thuyền đang chuyển động trên một con sông'],
 'segment_caption_vi': ['ba chiếc thuyền đang di_chuyển ở trên con sông',
  'có ba con thuyền đang di_chuyển trên con sông',
  'trên dòng sông có ba con thuyền đang di_chuyển',
  'ba con thuyền đang di_chuyển bên một cánh đồng lúa',
  'ba chiếc thuyền đang chuyển_động trên một con sông']}

# Tổng quan bộ DL
- DL gồm ảnh và caption (vietnamese) để huấn luyện cho bài toán Image caption
- Mỗi ảnh được lưu trữ ở kiểu DL PIL và mỗi ảnh có nhiều caption

In [4]:
len(dataset['train'])

3769

In [5]:
ROOT_DIR = Path.cwd().parent
ROOT_DIR

WindowsPath('D:/Code Module 6/VietNamese_Image_Captioning_with_CNN_LTSM')

In [6]:
SOURCE_DIR = ROOT_DIR/'source/'

In [7]:
sys.path.extend([str(ROOT_DIR), str(SOURCE_DIR)])

In [8]:
from organize_data import *

In [9]:
vicaptiondataset = VicaptioningDataSet(dataset=dataset, split='train',val_split=0.2,is_shuffle=True)

In [10]:
vicaptiondataset[0][0].shape

torch.Size([3, 224, 224])

In [11]:
len(vicaptiondataset)

15070

In [12]:
from vocabulary.vocab import Tokenizer, BuiltVocabFromIterator
from preprocessing.preprocessing_text import Preprocessing

In [13]:
processor = Preprocessing()

In [14]:
tokenizer = Tokenizer()

In [15]:
vicaptionDataLoader = DataLoader(dataset=vicaptiondataset, batch_size=512)

In [16]:
def yield_tokens(list_documentation):
    for doc in list_documentation:
        doc = processor.tranfer2Lower(processor.remove_punctuation_digit(doc))
        yield tokenizer(doc)

In [17]:
# get list caption of all data
documentations = tuple() 
for _, captions in vicaptionDataLoader:
    documentations +=captions

In [18]:
# build vocab
vocab = BuiltVocabFromIterator(
    iterator= yield_tokens(documentations),vocab_size=600,special_tokens=['<padd>', '<start>', '<end>', '<unk>']
)
del documentations
del vicaptionDataLoader

In [19]:
vocab.set_default(key='<unk>')

In [20]:
vocab.stoi()

{'<padd>': 0,
 '<start>': 1,
 '<end>': 2,
 '<unk>': 3,
 'có': 4,
 'một': 5,
 'ở': 6,
 'đang': 7,
 'người': 8,
 'hiện': 9,
 'xuất': 10,
 'trên': 11,
 'của': 12,
 'hàng': 13,
 'bên': 14,
 'nhiều': 15,
 'trong': 16,
 'chiếc': 17,
 'những': 18,
 'trước': 19,
 'nữ': 20,
 'áo': 21,
 'màu': 22,
 'phụ': 23,
 'mặc': 24,
 'được': 25,
 'đứng': 26,
 'hai': 27,
 'xe': 28,
 'đường': 29,
 'phía': 30,
 'gái': 31,
 'cô': 32,
 'sự': 33,
 'nhà': 34,
 'khu': 35,
 'đàn': 36,
 'ông': 37,
 'bức': 38,
 'cửa': 39,
 'là': 40,
 'ảnh': 41,
 'cái': 42,
 'quầy': 43,
 'con': 44,
 'cây': 45,
 'vài': 46,
 'mặt': 47,
 'bày': 48,
 'di': 49,
 'bán': 50,
 'tay': 51,
 'hình': 52,
 'xanh': 53,
 'ngồi': 54,
 'máy': 55,
 'đây': 56,
 'ngôi': 57,
 'và': 58,
 'chuyển': 59,
 'khung': 60,
 'trắng': 61,
 'chợ': 62,
 'đi': 63,
 'chụp': 64,
 'cảnh': 65,
 'thị': 66,
 'siêu': 67,
 'các': 68,
 'đồ': 69,
 'này': 70,
 'đỏ': 71,
 'ăn': 72,
 'cầm': 73,
 'kệ': 74,
 'phố': 75,
 'vào': 76,
 'đen': 77,
 'nhóm': 78,
 'căn': 79,
 'nhau': 80,
 'lá

In [21]:
CONFIGURATION_DIR = ROOT_DIR/'configuration'

In [22]:
sys.path.append(str(CONFIGURATION_DIR))

In [23]:
PATH_SAVE_VOCAB = CONFIGURATION_DIR/'vocab.json'

In [24]:
vocab.save(str(PATH_SAVE_VOCAB))

Save success vocab at D:\Code Module 6\VietNamese_Image_Captioning_with_CNN_LTSM\configuration\vocab.json


In [25]:
sentence_test = 'toi rat thích xem phim'

In [26]:
tokenizer(sentence_test)

['toi', 'rat', 'thích', 'xem', 'phim']

In [27]:
vocab(tokenizer(sentence_test))

[3, 3, 3, 306, 3]

In [28]:
vocab.itos()

{0: '<padd>',
 1: '<start>',
 2: '<end>',
 3: '<unk>',
 4: 'có',
 5: 'một',
 6: 'ở',
 7: 'đang',
 8: 'người',
 9: 'hiện',
 10: 'xuất',
 11: 'trên',
 12: 'của',
 13: 'hàng',
 14: 'bên',
 15: 'nhiều',
 16: 'trong',
 17: 'chiếc',
 18: 'những',
 19: 'trước',
 20: 'nữ',
 21: 'áo',
 22: 'màu',
 23: 'phụ',
 24: 'mặc',
 25: 'được',
 26: 'đứng',
 27: 'hai',
 28: 'xe',
 29: 'đường',
 30: 'phía',
 31: 'gái',
 32: 'cô',
 33: 'sự',
 34: 'nhà',
 35: 'khu',
 36: 'đàn',
 37: 'ông',
 38: 'bức',
 39: 'cửa',
 40: 'là',
 41: 'ảnh',
 42: 'cái',
 43: 'quầy',
 44: 'con',
 45: 'cây',
 46: 'vài',
 47: 'mặt',
 48: 'bày',
 49: 'di',
 50: 'bán',
 51: 'tay',
 52: 'hình',
 53: 'xanh',
 54: 'ngồi',
 55: 'máy',
 56: 'đây',
 57: 'ngôi',
 58: 'và',
 59: 'chuyển',
 60: 'khung',
 61: 'trắng',
 62: 'chợ',
 63: 'đi',
 64: 'chụp',
 65: 'cảnh',
 66: 'thị',
 67: 'siêu',
 68: 'các',
 69: 'đồ',
 70: 'này',
 71: 'đỏ',
 72: 'ăn',
 73: 'cầm',
 74: 'kệ',
 75: 'phố',
 76: 'vào',
 77: 'đen',
 78: 'nhóm',
 79: 'căn',
 80: 'nhau',
 81: